## Install Dependencies

In [ ]:
!pip install GEOparse bioservices mygene

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.3/145.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.1/720.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing ins

In [ ]:
import numpy as np
import pandas as pd

from bioservices import KEGG
import re
import random
import GEOparse
from itertools import islice
from collections import defaultdict, Counter

Creating directory /root/.config/bioservices 


## GSE48558 Dataset from GEO

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE25136

In [ ]:
# Download and parse the dataset
gse = GEOparse.get_GEO("GSE25136", destdir=".", silent=True)

In [ ]:
print(gse)

<SERIES: GSE25136 - 79 SAMPLES, 1 d(s)>


In [ ]:
# List all sample IDs
print(gse.gsms.keys())

# Inspect the first sample’s metadata
first_sample = list(gse.gsms.values())[0]
print(first_sample.metadata)

dict_keys(['GSM617581', 'GSM617582', 'GSM617583', 'GSM617584', 'GSM617585', 'GSM617586', 'GSM617587', 'GSM617588', 'GSM617589', 'GSM617590', 'GSM617591', 'GSM617592', 'GSM617593', 'GSM617594', 'GSM617595', 'GSM617596', 'GSM617597', 'GSM617598', 'GSM617599', 'GSM617600', 'GSM617601', 'GSM617602', 'GSM617603', 'GSM617604', 'GSM617605', 'GSM617606', 'GSM617607', 'GSM617608', 'GSM617609', 'GSM617610', 'GSM617611', 'GSM617612', 'GSM617613', 'GSM617614', 'GSM617615', 'GSM617616', 'GSM617617', 'GSM617618', 'GSM617619', 'GSM617620', 'GSM617621', 'GSM617622', 'GSM617623', 'GSM617624', 'GSM617625', 'GSM617626', 'GSM617627', 'GSM617628', 'GSM617629', 'GSM617630', 'GSM617631', 'GSM617632', 'GSM617633', 'GSM617634', 'GSM617635', 'GSM617636', 'GSM617637', 'GSM617638', 'GSM617639', 'GSM617640', 'GSM617641', 'GSM617642', 'GSM617643', 'GSM617644', 'GSM617645', 'GSM617646', 'GSM617647', 'GSM617648', 'GSM617649', 'GSM617650', 'GSM617651', 'GSM617652', 'GSM617653', 'GSM617654', 'GSM617655', 'GSM617656', '

In [ ]:
# Build metadata list
meta_rows = []
pattern = re.compile(r"recurrence status\s*:\s*(.+)", flags=re.I)

for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", []) or []
    recurrence = None
    for c in chars:
        if not isinstance(c, str):
            continue
        m = pattern.search(c)
        if m:
            recurrence = m.group(1).strip()
            break

    src = chars
    if src:
        source_name = src[0]
    else:
        title = gsm.metadata.get("title", [""])[0]
        source_name = title or "Unknown"

    meta_rows.append({
        "sample_id": gsm_name,
        "source_name": source_name,
        "recurrence_status": recurrence
    })

meta_df = pd.DataFrame(meta_rows).set_index("sample_id")
print("Recurrence counts:\n", meta_df["recurrence_status"].value_counts(dropna=False))

Recurrence counts:
 recurrence_status
Non-Recurrent    40
Recurrent        39
Name: count, dtype: int64


In [ ]:
# build expression dataframe (samples x probes)
expr_df = pd.DataFrame({
    gsm_name: gsm.table.set_index("ID_REF")["VALUE"].astype(float)
    for gsm_name, gsm in gse.gsms.items()
}).transpose()

print("Total samples in expr_df:", expr_df.shape[0], "probes:", expr_df.shape[1])

Total samples in expr_df: 79 probes: 22283


In [ ]:
# Map recurrence classes to numeric labels
label_map = {
    "Non-Recurrent": 0,
    "Recurrent": 1
}

# Create label series aligned with expr_df index
labels = meta_df.loc[expr_df.index, "recurrence_status"].map(label_map)

# Add label column as the last column
expr_df_labeled = expr_df.assign(label=labels.values)

# Ensure label is int
expr_df_labeled["label"] = expr_df_labeled["label"].astype(int)

print(expr_df_labeled.shape)
expr_df_labeled.head(3)

(79, 22284)


ID_REF,1007_s_at,1053_at,117_at,121_at,1255_g_at,1294_at,1316_at,1320_at,1405_i_at,1431_at,...,AFFX-r2-Hs28SrRNA-M_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,AFFX-TrpnX-3_at,AFFX-TrpnX-5_at,AFFX-TrpnX-M_at,label
GSM617581,797.8,2184.6,755.5,1770.1,1473.3,2210.6,11673.3,19668.4,27450.7,5.5,...,149.9,95.2,83.3,29.8,629.2,91.5,64.3,19.1,12.3,0
GSM617582,764.4,1883.4,625.3,1642.2,1547.5,2066.7,10840.9,17380.7,24178.7,10.9,...,117.6,62.0,137.6,84.2,370.3,47.0,121.0,16.6,21.7,0
GSM617583,414.4,1147.9,437.4,1137.3,941.8,1254.2,7353.7,12535.4,16988.0,12.4,...,115.8,77.6,77.1,27.5,331.6,92.6,97.9,10.6,13.4,1


In [ ]:
expr_df_labeled.to_csv("original.csv", index=True)
print(f"Saved to csv.")

Saved to csv.


## Map probe -> gene symbol

In [ ]:
def fetch_gpl_table(gpl_acc="GPL96", destdir="."):
    print(f"Downloading {gpl_acc} ...")
    gpl = GEOparse.get_GEO(geo=gpl_acc, destdir=destdir, silent=True)
    table = gpl.table
    print("GPL downloaded. Table shape:", table.shape)
    return table

def build_probe2symbol_from_table(table, probe_col, symbol_col):
    mapping = {}
    missing = 0
    for idx, row in table.iterrows():
        probe = str(row[probe_col]).strip()
        symbol_raw = row.get(symbol_col, "")
        if pd.isna(symbol_raw):
            mapping[probe] = None
            missing += 1
            continue
        symbol = str(symbol_raw).strip()
        # Clean symbol: some GPLs have 'SYMBOL /// Another' or 'GENE: SYMBOL (desc)'
        # try a few heuristics:
        # 1) split on '///' or '///' or ' // '
        symbol = re.split(r"\s*///\s*|\s*//\s*|;", symbol)[0]
        # 2) if the token contains spaces and likely is description, try to extract first token that looks like a gene symbol
        tokens = re.split(r"[,\s\(\)]+", symbol)
        # keep first token that matches typical gene symbol pattern (letters, digits, dash)
        sym_candidate = None
        for t in tokens:
            if re.match(r"^[A-Za-z0-9\-\._]+$", t):
                # skip tokens that are '---' or 'NA' or numeric only
                if t.upper() in ("NA", "---"):
                    continue
                if re.search(r"[A-Za-z]", t):
                    sym_candidate = t
                    break
        if sym_candidate is None:
            sym_candidate = symbol  # fallback to raw cleaned string
        mapping[probe] = sym_candidate
    print(f"Built mapping for {len(mapping)} probes ({missing} missing symbols).")
    return mapping

In [ ]:
gpl_table = fetch_gpl_table("GPL96")
probe_col = "ID"
symbol_col = "Gene Symbol"

GPL downloaded. Table shape: (22283, 16)


In [ ]:
probe2symbol = build_probe2symbol_from_table(gpl_table, probe_col, symbol_col)

print("Example of probe2symbol: ")
example1 = dict(islice(probe2symbol.items(), 5))
example1

Built mapping for 22283 probes (1058 missing symbols).
Example of probe2symbol: 


{'1007_s_at': 'DDR1',
 '1053_at': 'RFC2',
 '117_at': 'HSPA6',
 '121_at': 'PAX8',
 '1255_g_at': 'GUCA1A'}

In [ ]:
dataset_genes = list(set(probe2symbol.values()))
print("Number of unique symbols:", len(dataset_genes))

Number of unique symbols: 13238


In [ ]:
def map_col2gene_df(df, probe2symbol, label_col):

    df = df.copy()

    # identify probe columns in the dataframe that exist in probe2symbol
    probes_in_df = [c for c in df.columns if c in probe2symbol]
    # drop probes with missing symbol mapping
    probes_with_symbol = [p for p in probes_in_df if probe2symbol.get(p) is not None]

    # compose subset columns to keep
    keep_cols = probes_with_symbol[:]
    keep_cols.append(label_col)

    df = df.loc[:, keep_cols]

    # rename probes to gene symbols
    rename_map = {probe: probe2symbol[probe] for probe in probes_with_symbol}
    df = df.rename(columns=rename_map)

    # collapse probes that map to the same symbol by mean
    expr = df.drop(columns=[label_col])
    cancer_series = df[label_col]
    expr = expr.groupby(expr.columns, axis=1).mean()

    # reattach cancer column
    final_df = expr.copy()
    final_df[label_col] = cancer_series.reindex(final_df.index)

    return final_df

In [ ]:
gene_symbol_df = map_col2gene_df(expr_df_labeled, probe2symbol, label_col="label")

print(gene_symbol_df.shape)
gene_symbol_df.head(5)

(79, 13238)


/tmp/ipython-input-162475842.py:23: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  expr = expr.groupby(expr.columns, axis=1).mean()


ID_REF,A1CF,A2M,A4GALT,A4GNT,AAAS,AACS,AADAC,AAGAB,AAK1,AAMDC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYX,ZZEF1,ZZZ3,abParts,label
GSM617581,199.1,22.1,176.5,80.2,12.1,608.7,8.8,1291.50,361.22,330.666667,...,1445.3,458.7,268.65,31.6,1062.9,665.00,297.166667,547.1,178.8,0
GSM617582,129.1,10.3,251.0,18.4,10.8,557.6,7.7,469.05,189.64,377.900000,...,2073.7,491.1,376.50,34.7,1123.0,613.75,176.133333,429.7,145.8,0
GSM617583,121.0,9.6,362.9,123.1,52.0,534.4,87.1,1186.10,179.72,399.266667,...,1622.1,717.0,261.40,70.6,1568.3,492.20,222.266667,798.6,89.9,1
GSM617584,206.3,22.5,334.1,18.7,23.9,346.4,21.6,3253.95,319.66,269.866667,...,1630.0,497.4,304.65,19.7,827.0,619.30,203.733333,600.0,206.4,1
GSM617585,141.6,20.3,299.3,45.4,45.5,507.1,42.6,1342.85,240.14,179.566667,...,1930.3,455.5,203.60,115.0,1051.7,675.05,177.466667,1104.4,7.8,1


In [ ]:
gene_symbol_df.to_csv("original_mapped.csv", index=True)
print("saved csv")

saved csv


In [ ]:
gene_symbols = {c for c in gene_symbol_df.columns if c != "label"}

## Get genes from KEGG cancer pathways

In [ ]:
unique_intersect_gene = set()

In [ ]:
def fetch_kegg_aml_symbols(pathway):
    k = KEGG()
    entry = k.get(pathway)
    parsed = k.parse(entry)
    symbols = set()
    if 'GENE' in parsed:
        for entrez, desc in parsed['GENE'].items():
            m = re.match(r"^([A-Za-z0-9\-\._]+)", desc)
            if m:
                symbols.add(m.group(1).upper().strip())
    return symbols

def add_to_set(list):
    for item in list:
      if item not in unique_intersect_gene:
        unique_intersect_gene.add(item)
      pass

def get_intersect_genes(gene_symbols, pathway):
    symbols_in_pathway = fetch_kegg_aml_symbols(pathway)
    print(f'Number of symbols in {pathway}: {len(symbols_in_pathway)}')

    intersected_genes = sorted(list(gene_symbols.intersection(symbols_in_pathway)))
    print("Matched genes:", len(intersected_genes))

    add_to_set(intersected_genes)
    print("Number of intersected gene: ", len(unique_intersect_gene))

In [ ]:
kegg_pathway_list = [
    'hsa05200',
    'hsa05205',
    'hsa05206',
    'hsa05202',
    'hsa05223',
    'hsa05230',
    'hsa05222',
    'hsa05224',
    'hsa05215',
    'hsa05213',
    'hsa05226',
    'hsa05210',
    'hsa05212',
    'hsa05216',
    'hsa05219',
    'hsa05231',
    'hsa05235',
    'hsa05217',
    'hsa05208',
    'hsa05203',
    'hsa05169',
    'hsa05211',
    'hsa05204',
    'hsa05225',
    'hsa05207',
    'hsa05218',
    'hsa05214',
    'hsa05165',
    'hsa04371',
    'hsa03460',
    'hsa03320'
]

In [ ]:
for i in kegg_pathway_list:
  get_intersect_genes(gene_symbols, i)

Creating directory /root/.cache/bioservices 
Welcome to Bioservices
It looks like you do not have a configuration file.
We are creating one with default values in /root/.config/bioservices/bioservices.cfg .
Done
Number of symbols in hsa05200: 533
Matched genes: 488
Number of intersected gene:  488
Number of symbols in hsa05205: 204
Matched genes: 187
Number of intersected gene:  571
Number of symbols in hsa05206: 320
Matched genes: 149
Number of intersected gene:  632
Number of symbols in hsa05202: 201
Matched genes: 166
Number of intersected gene:  738
Number of symbols in hsa05223: 73
Matched genes: 70
Number of intersected gene:  743
Number of symbols in hsa05230: 71
Matched genes: 66
Number of intersected gene:  772
Number of symbols in hsa05222: 93
Matched genes: 91
Number of intersected gene:  772
Number of symbols in hsa05224: 148
Matched genes: 132
Number of intersected gene:  777
Number of symbols in hsa05215: 106
Matched genes: 101
Number of intersected gene:  795
Number of s

In [ ]:
with open("kegg_intersect.txt", "w") as f:
  f.write(repr(unique_intersect_gene))

## Extract subset from intersected genes

In [ ]:
df = pd.read_csv('original_mapped.csv')
df.head(5)

,Unnamed: 0,A1CF,A2M,A4GALT,A4GNT,AAAS,AACS,AADAC,AAGAB,AAK1,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYX,ZZEF1,ZZZ3,abParts,label
0,GSM617581,199.1,22.1,176.5,80.2,12.1,608.7,8.8,1291.50,361.22,...,1445.3,458.7,268.65,31.6,1062.9,665.00,297.166667,547.1,178.8,0
1,GSM617582,129.1,10.3,251.0,18.4,10.8,557.6,7.7,469.05,189.64,...,2073.7,491.1,376.50,34.7,1123.0,613.75,176.133333,429.7,145.8,0
2,GSM617583,121.0,9.6,362.9,123.1,52.0,534.4,87.1,1186.10,179.72,...,1622.1,717.0,261.40,70.6,1568.3,492.20,222.266667,798.6,89.9,1
3,GSM617584,206.3,22.5,334.1,18.7,23.9,346.4,21.6,3253.95,319.66,...,1630.0,497.4,304.65,19.7,827.0,619.30,203.733333,600.0,206.4,1
4,GSM617585,141.6,20.3,299.3,45.4,45.5,507.1,42.6,1342.85,240.14,...,1930.3,455.5,203.60,115.0,1051.7,675.05,177.466667,1104.4,7.8,1


In [ ]:
with open("kegg_intersect.txt") as f:
    initial_intersect = eval(f.read())

print(len(initial_intersect))
print(list(initial_intersect)[:10])

1431
['TSC2', 'RDX', 'LYN', 'IDH2', 'FZD1', 'GTF2A1L', 'EFNA5', 'HMGA2', 'EP300', 'PPP2R5A']


In [ ]:
df.columns = [str(c).strip() for c in df.columns]
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [ ]:
df.head()

,A1CF,A2M,A4GALT,A4GNT,AAAS,AACS,AADAC,AAGAB,AAK1,AAMDC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYX,ZZEF1,ZZZ3,abParts,label
0,199.1,22.1,176.5,80.2,12.1,608.7,8.8,1291.50,361.22,330.666667,...,1445.3,458.7,268.65,31.6,1062.9,665.00,297.166667,547.1,178.8,0
1,129.1,10.3,251.0,18.4,10.8,557.6,7.7,469.05,189.64,377.900000,...,2073.7,491.1,376.50,34.7,1123.0,613.75,176.133333,429.7,145.8,0
2,121.0,9.6,362.9,123.1,52.0,534.4,87.1,1186.10,179.72,399.266667,...,1622.1,717.0,261.40,70.6,1568.3,492.20,222.266667,798.6,89.9,1
3,206.3,22.5,334.1,18.7,23.9,346.4,21.6,3253.95,319.66,269.866667,...,1630.0,497.4,304.65,19.7,827.0,619.30,203.733333,600.0,206.4,1
4,141.6,20.3,299.3,45.4,45.5,507.1,42.6,1342.85,240.14,179.566667,...,1930.3,455.5,203.60,115.0,1051.7,675.05,177.466667,1104.4,7.8,1


In [ ]:
# keep only intersected genes that are present in the df
genes_present = [g for g in initial_intersect if g in df.columns]
missing_genes = [g for g in initial_intersect if g not in df.columns]
cols_to_keep = genes_present + ["label"]

print(f"Intersected genes found in dataframe: {len(genes_present)}")

Intersected genes found in dataframe: 1431


In [ ]:
reduced_df = df.loc[:, cols_to_keep].copy()

print("Reduced dataframe shape (rows x cols):", reduced_df.shape)
print("Label distribution in reduced set:")
print(reduced_df["label"].value_counts())

Reduced dataframe shape (rows x cols): (79, 1432)
Label distribution in reduced set:
label
0    40
1    39
Name: count, dtype: int64


In [ ]:
reduced_df.to_csv("reduced.csv", index=False)
print(f"Saved reduced dataframe")

Saved reduced dataframe
